# 05 — Compare Methods

Side-by-side comparison of SAE / JLens / patching on several culture prompt pairs.
Produces a summary table + figures under `results/comparison/`.

In [ ]:
# --- Colab / local bootstrap (Drive + wheels + swappable model) ---
# Drive layout expected:
#   MyDrive/multilingual-mechinterp/
#     dist/*.whl
#     data/all200questions_persianMiddleEastCulture.json
#     configs/  notebooks/  results/
#
# Edit MODEL_NAME in notebooks/colab_setup.py (Qwen2.5 now; Gemma later),
# or override below after bootstrap.

from pathlib import Path
import runpy

def _resolve_setup_script() -> Path:
    here = Path.cwd()
    candidates = [
        here / "colab_setup.py",
        here / "notebooks" / "colab_setup.py",
        here.parent / "notebooks" / "colab_setup.py",
        Path("/content/drive/MyDrive/multilingual-mechinterp/notebooks/colab_setup.py"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "colab_setup.py not found. Mount Drive with the project folder, "
        "or open the notebook from the repo."
    )

_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

# Session overrides (uncomment as needed):
# MODEL_NAME = "google/gemma-2-2b"
# MODEL_TRUST_REMOTE_CODE = False
# USE_TINY_OFFLINE = True   # demos without downloading HF weights

import matplotlib.pyplot as plt
import torch

from multilingual_mechinterp.utils import ensure_dir, load_config

cfg_path = CONFIG_DIR / "qwen25.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}
if "model" in cfg and not USE_TINY_OFFLINE:
    # keep notebook MODEL_NAME as source of truth; cfg is fallback metadata
    pass

print("Ready.")
print(" ROOT =", ROOT)
print(" DATA =", DATA_DIR)
print(" DIST =", DIST_DIR)
print(" MODEL =", MODEL_NAME, "| tiny=", USE_TINY_OFFLINE)

from multilingual_mechinterp.data import culture_prompt_pairs, load_culture_questions
from multilingual_mechinterp.jlens import TinyDecoder, fit, run_jlens
from multilingual_mechinterp.metrics import evaluate_sae
from multilingual_mechinterp.patching import TinyCausalLM, recommend_layers, run_patching
from multilingual_mechinterp.sae import train_tied_sae
from multilingual_mechinterp.utils import save_json

OUT = ensure_dir(RESULTS_DIR / "comparison")
N_ITEMS = 5


## Load experiment model

Uses `MODEL_NAME` from `colab_setup.py` (default **Qwen2.5**). Set `USE_TINY_OFFLINE=True` for demos without HF downloads.


In [ ]:
# Real model (Qwen now; change MODEL_NAME for Gemma later) OR tiny offline
# model = load_experiment_model()
# For gated Gemma: export HF_TOKEN=... or pass token=...

# Default path in analysis cells below uses tiny models for speed.
# Swap in `model = load_experiment_model()` when you are ready for Qwen/Gemma.
print("To load HF weights:", f"load_experiment_model({MODEL_NAME!r})")
print("Culture JSON:", culture_json_path(), "exists=", culture_json_path().exists())


## 1. Shared offline models + culture pairs

In [ ]:
path = ROOT / "data" / "all200questions_persianMiddleEastCulture.json"
if path.exists():
    pairs = culture_prompt_pairs(
        load_culture_questions(path, limit=N_ITEMS),
        source_lang="english",
        target_lang="persian",
        limit=N_ITEMS,
    )
else:
    pairs = [
        {
            "id": str(i),
            "source_prompt": f"Fact {i}: The capital of France is",
            "target_prompt": f"Fact {i}: پایتخت فرانسه",
            "source_answer": "Paris",
            "target_answer": "پاریس",
        }
        for i in range(N_ITEMS)
    ]

# Fit shared offline demos once
torch.manual_seed(0)
acts = torch.randn(3000, 64)
sae = train_tied_sae(acts, ratio=2, alpha=8.6e-4, n_epochs=1, batch_size=256).sae
sae_metrics = evaluate_sae(sae, acts[:1500])

jlens_model = TinyDecoder(n_layers=4, d_model=16, seed=0)
lens = fit(
    jlens_model,
    [p["source_prompt"][:90] for p in pairs] + ["Nowruz Hafez culture tradition Iran"],
    source_layers=[0, 1, 2],
    target_layer=3,
    skip_first=0,
    dim_batch=8,
)
patch_model = TinyCausalLM(n_layers=6, d_model=32, seed=0)
print("pairs=", len(pairs), "sae_fvu=", round(sae_metrics["fvu"], 4))

## 2. Per-item comparison

In [ ]:
rows = []
effect_matrix = []  # items x layers

for pair in pairs:
    src = pair["source_prompt"][:160]
    tgt = pair["target_prompt"][:160]
    ans = str(pair["source_answer"]).split()[0]

    j = run_jlens(jlens_model, src, lens=lens, layers=[0, 1, 2], positions=[-1], top_k=3)
    p = run_patching(patch_model, src, tgt, answer=ans)
    top_layers = recommend_layers(p, top_k=2, min_effect=-1.0)

    with torch.no_grad():
        # proxy: encode random residual as stand-in for prompt features
        c = sae.encode(torch.randn(1, 64))
        top_feat = int(torch.topk(c.squeeze(0), k=1).indices[0])

    rows.append({
        "id": pair["id"],
        "answer": ans,
        "sae_top_feature": top_feat,
        "jlens_top_L2": j.top_tokens.get(2, [{}])[0].get("token"),
        "patch_best_layer": p.best_layer,
        "patch_max_effect": max(p.scores.values()) if p.scores else None,
        "layers_to_swap": top_layers,
    })
    effect_matrix.append([p.scores[L] for L in sorted(p.scores)])

save_json(rows, OUT / "method_comparison.json")
rows

In [ ]:
import numpy as np

M = np.array(effect_matrix)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im = axes[0].imshow(M, aspect="auto", cmap="coolwarm")
axes[0].set_xlabel("start layer")
axes[0].set_ylabel("item index")
axes[0].set_title("Patching ΔP heatmap (items × layers)")
fig.colorbar(im, ax=axes[0], fraction=0.046)

best_layers = [r["patch_best_layer"] for r in rows]
axes[1].hist(best_layers, bins=range(0, M.shape[1] + 1), color="#4C78A8", edgecolor="white")
axes[1].set_xlabel("best start layer")
axes[1].set_ylabel("count")
axes[1].set_title("Where to swap? (mode of best layers)")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig.savefig(OUT / "compare_methods.png", dpi=150)
plt.show()

print("SAE global FVU/L0:", sae_metrics["fvu"], sae_metrics["mean_l0"])
print("Recommended swap layers (union):",
      sorted({L for r in rows for L in r["layers_to_swap"]}))

## 3. Next step on Colab with a real LLM

1. `pip install -e .`
2. `load_model("EleutherAI/pythia-70m-deduped")` (or Gemma if you have access)
3. Train SAE on residual extracts; fit JLens on webtext; run `run_patching` on culture pairs
4. Reuse `recommend_layers` to pick intervention sites for SAE feature / JLens concept edits